In [2]:
pip install datasets

In [3]:
from datasets import load_dataset

ds = load_dataset("Teklia/IAM-line")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train.parquet:   0%|          | 0.00/167M [00:00<?, ?B/s]

data/validation.parquet:   0%|          | 0.00/24.7M [00:00<?, ?B/s]

data/test.parquet:   0%|          | 0.00/73.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6482 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/976 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2915 [00:00<?, ? examples/s]

In [4]:
# Cell 4 — Save text to file
with open("handwriting_dataset.txt", "w", encoding="utf-8") as f:
    for sample in ds['train']:
        f.write(sample['text'] + "\n")

print("✅ Saved handwriting_dataset.txt")


✅ Saved handwriting_dataset.txt


In [5]:
# Cell 5 — Read the saved file and preview
with open("handwriting_dataset.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Total characters : {len(text)}")
print(f"Total lines      : {text.count(chr(10))}")
print(f"\nFirst 500 characters preview:")
print(text[:500])

Total characters : 287727
Total lines      : 6482

First 500 characters preview:
put down a resolution on the subject
and he is to be backed by Mr. Will
nominating any more Labour life Peers
M Ps tomorrow. Mr. Michael Foot has
Griffiths, M P for Manchester Exchange .
is to be made at a meeting of Labour
A MOVE to stop Mr. Gaitskell from
0M P for Manchester Exchange .
A MOVE to stop Mr. Gaitskell from nominating
meeting of Labour 0M Ps tommorow . Mr. Michael
any more Labour life Peers is to be made at a
Foot has put down a resolution on the subject
and he is to be backed by M


In [6]:
# Cell 6 — Build character vocabulary
chars = sorted(set(text))
vocab_size = len(chars)

char2idx = {ch: i for i, ch in enumerate(chars)}
idx2char  = {i: ch for i, ch in enumerate(chars)}

print(f"Unique characters : {vocab_size}")
print(f"Characters        : {''.join(chars)}")

Unique characters : 80
Characters        : 
 !"#&'()*+,-./0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [7]:
# Cell 7 — Dataset class
import torch
from torch.utils.data import Dataset, DataLoader

class HandwritingDataset(Dataset):
    def __init__(self, text, char2idx, seq_length=100):
        self.seq_length = seq_length
        self.data = torch.tensor([char2idx[ch] for ch in text], dtype=torch.long)

    def __len__(self):
        return len(self.data) - self.seq_length

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_length]
        y = self.data[idx + 1 : idx + self.seq_length + 1]
        return x, y

dataset_obj = HandwritingDataset(text, char2idx, seq_length=100)
loader      = DataLoader(dataset_obj, batch_size=64, shuffle=True)

print(f"Total training samples : {len(dataset_obj)}")

Total training samples : 287627


In [8]:
# Cell 8 — RNN Model
import torch.nn as nn

class HandwritingRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_size=512, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm       = nn.LSTM(embed_dim, hidden_size, num_layers,
                                  dropout=0.3, batch_first=True)
        self.dropout    = nn.Dropout(0.3)
        self.fc         = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x, hidden = self.lstm(self.embedding(x), hidden)
        return self.fc(self.dropout(x)), hidden

    def init_hidden(self, batch_size, device):
        h = torch.zeros(2, batch_size, 512).to(device)
        c = torch.zeros(2, batch_size, 512).to(device)
        return (h, c)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = HandwritingRNN(vocab_size).to(device)

print(f"Using device      : {device}")
print(f"Model parameters  : {sum(p.numel() for p in model.parameters()):,}")

Using device      : cuda
Model parameters  : 3,467,344


In [10]:
# Cell 9 — Training
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        hidden = model.init_hidden(x.size(0), device)

        optimizer.zero_grad()
        logits, _ = model(x, hidden)

        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}  |  Loss: {total_loss/len(loader):.4f}")

Epoch 1/5  |  Loss: 0.6124
Epoch 2/5  |  Loss: 0.3548
Epoch 3/5  |  Loss: 0.3073
Epoch 4/5  |  Loss: 0.2867
Epoch 5/5  |  Loss: 0.2743


In [11]:
# Cell 10 — Generate handwritten-style text
import torch.nn.functional as F

def generate(seed_text, length=300, temperature=0.8):
    model.eval()
    input_seq = torch.tensor(
        [char2idx[ch] for ch in seed_text if ch in char2idx],
        dtype=torch.long
    ).unsqueeze(0).to(device)

    hidden    = model.init_hidden(1, device)
    generated = seed_text

    with torch.no_grad():
        for _ in range(length):
            logits, hidden = model(input_seq, hidden)
            probs          = F.softmax(logits[0, -1, :] / temperature, dim=-1)
            next_idx       = torch.multinomial(probs, 1).item()
            generated     += idx2char[next_idx]
            input_seq      = torch.tensor([[next_idx]], dtype=torch.long).to(device)

    return generated

# ✅ Try it!
print(generate("the quick", length=300, temperature=0.8))

the quick in-toming
of individual instrumental timbres and " personal "
The libretto , by W. H. Auden and Chester Kallman ,
surrounding an ageing poet , whose deeply
mischief perpetrated by almost every person
the bones Doris Langley Moore has brought
who had been close to him . In turning over
behaviour and


In [ ]:
from google.colab import files
files.upload()

In [15]:
import nbformat

with open("tet.ipynb", "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

# Remove widgets metadata
if "widgets" in nb["metadata"]:
    del nb["metadata"]["widgets"]

with open("text.ipynb", "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

FileNotFoundError: [Errno 2] No such file or directory: 'tet.ipynb'